# Trabajo Práctico Integrador — Expensa House

## Propósito: inferencia, no entrenamiento

En este trabajo **no entrenamos una IA ni hacemos fine-tuning**: usamos inferencia con modelos preentrenados y publicados en Hugging Face. Elegimos como problema principal buscar ofertas cargadas manualmente; E5 calcula similitud semántica para ordenarlas, pero el sistema también exige coincidencia de palabras clave antes de comparar precios. El score no es una probabilidad ni confirma que marca, variedad, tamaño y unidad sean idénticos.

## Enlaces del TP

- Web local de Expensa House: [http://127.0.0.1:8001/](http://127.0.0.1:8001/). Solo funciona si el servidor está iniciado en esta computadora; **no es una URL pública**.
- Repositorio y rama: [Seminario2026 — Proyecto_Integrador](https://github.com/griseldablanco78-blip/Seminario2026/tree/Proyecto_Integrador).
- Carpeta del TP: [Clase 7/TP_Integrador](https://github.com/griseldablanco78-blip/Seminario2026/tree/Proyecto_Integrador/Clase%207/TP_Integrador).
- Model cards: [MiniLM multilingüe](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) y [E5 multilingüe small](https://huggingface.co/intfloat/multilingual-e5-small).
- Consigna completa: [Clase 07 — Consigna del TP Integrador](https://aulasvirtuales.bue.edu.ar/mod/resource/view.php?id=1063309) (puede requerir iniciar sesión).
- URL pública de la app, informe PDF y presentación PPTX: pendientes.

## Los cinco pasos solicitados

1. **Elegir problema y modelos.** El comparador responde a una necesidad de Expensa House y permite evaluar nombres escritos de distintas formas. Se buscaron modelos con la tarea `Sentence Similarity` y uso con Transformers, y se leyeron sus model cards. Se comparan E5 y MiniLM; ambos son multilingües y admiten español. E5 se eligió para integrar por su uso de recuperación consulta-documento y los prefijos recomendados `query:` y `passage:`. MiniLM se usa como segundo candidato, no está conectado a la web.
2. **Probar dos modelos en la notebook.** Las siguientes celdas comparan MiniLM y E5 con las mismas ofertas, consultas y etiquetas; prueban una línea base literal y `transformers.pipeline`, e intentan repetir la inferencia con `InferenceClient`. El proveedor remoto devolvió HTTP 402 por falta de créditos: no se obtuvieron scores remotos y no se inventan.
3. **Armar y publicar una app.** E5 está integrado al comparador de la web existente, que mantiene FastAPI/HTML/JavaScript y usa las ofertas manuales sincronizadas. La web solo está confirmada en local, en el enlace de arriba. La consigna menciona Gradio o Streamlit: la web original no usa esos frameworks. `TP_Integrador/app.py` es una demo Gradio independiente con datos inventados; no es la web sincronizada. La URL pública y el cumplimiento de ese requisito literal siguen pendientes.
4. **Ordenar la carpeta del TP.** `Clase 7/TP_Integrador/` contiene `app.py`, `requirements.txt`, `README.md`, un CSV didáctico y esta notebook. El CSV tiene ejemplos inventados y no son precios reales de Expensa House. Verificá que los archivos estén guardados y subidos a la rama enlazada arriba.
5. **Escribir informe y presentación.** Faltan `TP_Apellido_Nombre.pdf` y `TP_Apellido_Nombre.pptx`. Deben justificar dónde vive el código (GitHub), dónde viven los pesos preentrenados (Hugging Face), dónde se ejecuta la inferencia (localmente en la API; luego en el servidor que se publique), qué se midió y cuáles son las limitaciones. La entrega solicita PDF, notebook, PPTX, enlace público de la app y enlace a la carpeta del TP.


## Paso 1 — definir el problema y leer las model cards

Problema acotado: dada una consulta (por ejemplo, **“leche entera de un litro”**), ordenar ofertas cargadas con nombres que pueden estar escritos de otra manera. No entrenamos modelos: usamos inferencia de modelos preentrenados publicados en Hugging Face.

Comparamos dos modelos candidatos para español, encontrados al filtrar en Hugging Face por la tarea **Sentence Similarity** y la biblioteca **Transformers**.

- [`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2): modelo de embeddings de oraciones multilingüe; su ficha declara uso para búsqueda semántica y similitud. Es el candidato especializado en similitud de oraciones.
- [`intfloat/multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small): modelo de embeddings multilingüe con vectores de 384 dimensiones. La ficha recomienda los prefijos `query:` y `passage:` para recuperación consulta-catálogo. Es el modelo integrado en el comparador de la web principal.

### Lectura comparada de las model cards

Consulta de las fichas: La información que sigue resume lo declarado por cada tarjeta; “no detallado” significa que la ficha consultada no aporta ese dato de forma explícita.

| Aspecto | MiniLM multilingüe | E5 multilingüe small |
|---|---|---|
| Modelo y tarea | [`paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2); `sentence-similarity`, búsqueda semántica y agrupamiento. | [`multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small); `sentence-similarity`, embeddings y recuperación semántica. |
| Biblioteca declarada | `library_name: sentence-transformers`; la ficha incluye la sección “Usage (HuggingFace Transformers)” y una receta con `AutoTokenizer`/`AutoModel`. | `library_name: sentence-transformers`; la ficha contiene una receta de uso con `transformers.AutoTokenizer` y `transformers.AutoModel`. |
| Español e idiomas | Multilingüe; español (`es`) figura en la lista de idiomas de la ficha. | Multilingüe; español figura entre 100 idiomas. La ficha advierte que los idiomas con pocos recursos pueden rendir peor. |
| Arquitectura / representación | 12 capas MiniLM; vector de 384 dimensiones; longitud máxima indicada: 128 tokens. | 12 capas; vector de 384 dimensiones; entradas de hasta 512 tokens según el ejemplo de la ficha. |
| Entrenamiento y datos | La ficha indica que fue entrenado por Sentence Transformers y cita Sentence-BERT, pero no detalla claramente el corpus de entrenamiento de este modelo. | Inicializado desde `microsoft/Multilingual-MiniLM-L12-H384`; la ficha describe preentrenamiento contrastivo con supervisión débil y ajuste supervisado. Para más datos remite al informe técnico; no presenta allí una lista completa de corpus. |
| Licencia | Apache-2.0. | MIT. |
| Parámetros y tamaño relativo | La ficha no da el total; los metadatos `safetensors` del Hub informan 117.653.760 valores F32 (≈117,7 M). 384 es la dimensión del embedding, no el número de parámetros. | La ficha no da el total; los metadatos `safetensors` del Hub informan 117.653.760 valores F32 (≈117,7 M). 384 es la dimensión del embedding, no el número de parámetros. |
| Advertencias relevantes | Es un modelo de embeddings, no una decisión de equivalencia de productos. Debe calibrarse con ejemplos del dominio; límite de entrada de 128 tokens. | La similitud no confirma que marca, variante y cantidad sean iguales; usar los prefijos recomendados y considerar la advertencia sobre idiomas de pocos recursos. |

Los dos modelos se encuentran en el segmento “small” de esta familia, con unos 118 M de parámetros cada uno; no son modelos diminutos y la inferencia local puede tardar o requerir memoria. Las dos fichas respaldan el uso con Transformers, pero el nombre de biblioteca declarado es `sentence-transformers`. La notebook usa `transformers.pipeline` para extraer vectores; esto no cambia el metadato ni convierte el score coseno en una probabilidad. La disponibilidad de cada modelo en `InferenceClient` depende del proveedor y debe comprobarse al ejecutar: la ficha del Hub no garantiza que esté servido por `hf-inference`.


### Modelos de las tres problemáticas de Expensa House

| Problemática | Qué ya existe en Expensa House | Modelo del Hub apropiado y situación |
|---|---|---|
| Comparar ofertas | La web principal usa las ofertas manuales sincronizadas. E5 las ordena por similitud, pero el sistema solo compara precios si más de la mitad de las palabras clave de la consulta también aparece en el producto o marca. | Integrado en el comparador: [`intfloat/multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small), tarea `sentence-similarity`, 117.653.760 parámetros, biblioteca declarada `sentence-transformers` (la ficha también muestra uso con Transformers). MiniLM es el segundo modelo evaluado en esta notebook. |
| Pasar voz a texto | La web principal ya ofrece un botón de voz con Web Speech API del navegador; el endpoint servidor existente usa reconocimiento de Google, no un modelo HF. | [`openai/whisper-tiny`](https://huggingface.co/openai/whisper-tiny), tarea `automatic-speech-recognition`, 37.760.640 parámetros, biblioteca `transformers`, licencia Apache-2.0 e idioma español declarado. Está usado localmente en `TP_Integrador/app.py` separado; no está conectado a la web principal. Se puede integrar al endpoint de voz, sujeto a memoria y tiempo de descarga. |
| Chatbot | Boti usa reglas, no un modelo generativo. Las opciones del chat todavía requieren corrección y prueba; no se considera terminado. | Candidato no integrado: [`Qwen/Qwen2.5-0.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), tarea `text-generation`, 494.032.768 parámetros, biblioteca `transformers`, licencia Apache-2.0; su ficha declara español. Si se evalúa, stock y precios deben provenir de las funciones de datos existentes y no ser inventados por el modelo. |

La web principal **conserva la interfaz, los módulos y los datos sincronizados de la app**. E5 ya está integrado en el comparador: calcula similitud entre la consulta y las ofertas manuales, pero no alcanza con un score alto para afirmar que sea el mismo producto. Para reducir falsos positivos, la web exige además que más de la mitad de las palabras clave consultadas aparezca en el nombre o marca guardados. Esta política es conservadora: puede rechazar paráfrasis con vocabulario completamente distinto. E5 se carga al primer pedido que tenga ofertas compatibles, requiere Internet para descargar los pesos la primera vez y no necesita `HF_TOKEN`. Si la carga falla, la API devuelve un error 503 explícito. La prueba reproducible de esta notebook usa el CSV etiquetado de demostración y no lee ni reemplaza los registros reales. Whisper y Qwen quedan como candidatos para las otras dos problemáticas, no como modelos conectados a la web.

**Interfaz y publicación:** la web principal usa FastAPI con HTML/JavaScript; no es Gradio ni Streamlit. `TP_Integrador/app.py` es una demo Gradio separada con datos inventados, no la web sincronizada. La web se probó localmente en `http://127.0.0.1:8001/`, pero aún no tiene enlace público confirmado. No presentar como pública una dirección `127.0.0.1` ni afirmar que la demo independiente es la web integrada.

Estos modelos son preentrenados: en este TP los descargamos y usamos para inferencia; no los entrenamos ni ajustamos con los datos privados del hogar.

La evaluación local usa las mismas consultas y ofertas para ambos modelos. La evaluación remota con `InferenceClient` es un intento separado: si el proveedor no acepta un modelo, no hay créditos o la llamada falla, registrá el error como resultado y no inventes scores. La prueba actual se ejecutó con el CSV didáctico; no alcanza para elegir un ganador de producción. La decisión final debe basarse en datos representativos y consultas etiquetadas.

**Datos:** `ofertas.csv` es un conjunto pequeño de demostración para validar el experimento, no el inventario personal ni un feed real de supermercados. No se deben presentar sus precios como reales. Para validar Expensa House, usá ofertas manuales observadas y registrá producto, marca, supermercado, precio, cantidad, unidad, fecha y fuente.

## Paso 2 — preparar el entorno y los datos

Ejecutá las celdas en orden. En Colab, subí `ofertas.csv` cuando se solicite. En Jupyter local, abrí la notebook desde la carpeta raíz del repositorio, desde `TP_Integrador/notebook`, o dejá el CSV en la misma carpeta que esta notebook.

In [9]:
%pip install -q transformers torch huggingface_hub pandas numpy python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path
import os
import re
import time

import numpy as np
import pandas as pd

possible_paths = [
    Path("ofertas.csv"),
    Path("../ofertas.csv"),
    Path("TP_Integrador/ofertas.csv"),
    Path("../TP_Integrador/ofertas.csv"),
]
csv_path = next((path for path in possible_paths if path.is_file()), None)
if csv_path is None:
    try:
        from google.colab import files
        print("Subí ofertas.csv desde tu computadora.")
        files.upload()
        csv_path = Path("ofertas.csv")
    except ImportError as exc:
        raise FileNotFoundError(
            "No encontré ofertas.csv. Dejalo junto a la notebook o en TP_Integrador/."
        ) from exc

offers = pd.read_csv(csv_path)
required = {"producto", "familia", "precio", "cantidad", "unidad", "fecha_relevamiento", "fuente"}
missing = required.difference(offers.columns)
if missing:
    raise ValueError(f"Faltan columnas en ofertas.csv: {sorted(missing)}")
offers.head()

,supermercado,producto,familia,precio,cantidad,unidad,fecha_relevamiento,fuente
0,Super A,Leche entera La Serenisima sachet 1 litro,leche,1450,1000,ml,2026-10-01,Ejemplo inventado
1,Super B,Leche LS entera 1L,leche,1390,1000,ml,2026-10-01,Ejemplo inventado
2,Super C,Leche entera larga vida 1 lt,leche,1520,1000,ml,2026-10-01,Ejemplo inventado
3,Super A,Yerba mate Taragui 1kg,yerba,5200,1000,g,2026-10-01,Ejemplo inventado
4,Super B,Yerba Taragui con palo 1 kg,yerba,4990,1000,g,2026-10-01,Ejemplo inventado


Las familias del CSV son etiquetas conocidas para evaluar este conjunto pequeño. Incluimos consultas normales, paráfrasis, dos negativos y dos casos desafiantes: una consulta en inglés (`milk`) y una consulta muy corta (`té`). En una evaluación real, escribí las consultas y etiquetas antes de mirar las respuestas del modelo, y registrá cuáles deberían coincidir.

In [11]:
TEST_CASES = [
    ("leche entera de un litro", "leche"),
    ("leche para el desayuno", "leche"),
    ("yerba para mate, un kilo", "yerba"),
    ("yerba mate suave", "yerba"),
    ("detergente para lavar los platos", "detergente"),
    ("lavavajillas con limón", "detergente"),
    ("fideos tipo tirabuzón", "fideos"),
    ("arroz integral", "sin_coincidencia"),
    ("jabón para manos", "sin_coincidencia"),
    ("milk", "leche"),  # Otro idioma: misma familia, nombre de oferta en español.
    ("té", "sin_coincidencia"),  # Consulta muy corta y sin esa familia en el CSV.
]
PRODUCTS = offers["producto"].astype(str).tolist()
THRESHOLD = 0.50  # Inicial: calibrar con los falsos positivos/negativos observados.

## Paso 3 — línea base sin IA

Esta regla solo busca la consulta completa dentro del nombre. Es deliberadamente simple: sirve para comparar y explicar qué agrega la similitud semántica; no es un competidor sofisticado.

In [3]:
def keyword_baseline(query, products):
    query = query.casefold().strip()
    return next((product for product in products if query in product.casefold()), None)

baseline_rows = []
for query, expected in TEST_CASES:
    match = keyword_baseline(query, PRODUCTS)
    family = offers.loc[offers["producto"].eq(match), "familia"].iloc[0] if match else "sin_coincidencia"
    baseline_rows.append({"consulta": query, "esperado": expected, "resultado_regla": family, "producto": match})
baseline_df = pd.DataFrame(baseline_rows)
baseline_df

,consulta,esperado,resultado_regla,producto
0,leche entera de un litro,leche,sin_coincidencia,NaN
1,leche para el desayuno,leche,sin_coincidencia,NaN
2,"yerba para mate, un kilo",yerba,sin_coincidencia,NaN
3,yerba mate suave,yerba,yerba,Yerba mate suave 500g
4,detergente para lavar los platos,detergente,sin_coincidencia,NaN
5,lavavajillas con limón,detergente,sin_coincidencia,NaN
6,fideos tipo tirabuzón,fideos,sin_coincidencia,NaN
7,arroz integral,sin_coincidencia,sin_coincidencia,NaN
8,jabón para manos,sin_coincidencia,sin_coincidencia,NaN
9,milk,leche,sin_coincidencia,NaN


## Paso 4 — inferencia local con `transformers.pipeline`

`pipeline("feature-extraction")` entrega un vector por token, no directamente una decisión de “mismo producto”. Para obtener un score comparable, promediamos los vectores de tokens de cada texto y calculamos coseno entre la consulta y cada nombre. Esta simplificación sigue la operación de pooling que describen las fichas; no implica que el score sea una probabilidad.

El pipeline corre en CPU por defecto. La primera ejecución descarga los pesos, por lo que puede tardar.

In [16]:
import torch
import torch.nn.functional as F
from functools import lru_cache
from transformers import pipeline

MODEL_IDS = {
    "MiniLM-L12 multilingüe": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "E5 multilingüe small": "intfloat/multilingual-e5-small",
}

def format_input(model_id, text, role):
    if model_id == "intfloat/multilingual-e5-small":
        return f"{role}: {text}"
    return text

def build_local_scorer(model_id):
    extractor = pipeline("feature-extraction", model=model_id, device=-1)

    @lru_cache(maxsize=None)
    def encode(text, role):
        output = extractor(format_input(model_id, text, role), truncation=True, max_length=128)
        token_vectors = torch.as_tensor(output[0], dtype=torch.float32)
        pooled = token_vectors.mean(dim=0)
        return F.normalize(pooled, p=2, dim=0)

    def score(query, candidates):
        query_vector = encode(query, "query")
        candidate_vectors = torch.stack([encode(candidate, "passage") for candidate in candidates])
        scores = candidate_vectors @ query_vector
        return scores.detach().cpu().numpy()

    return extractor, score

def evaluate_local(model_id, model_name):
    extractor, score = build_local_scorer(model_id)
    rows = []
    started = time.perf_counter()
    first_output_type = None
    for query, expected in TEST_CASES:
        scores = score(query, PRODUCTS)
        best_index = int(np.argmax(scores))
        best = offers.iloc[best_index]
        best_score = float(scores[best_index])
        predicted = best["familia"] if best_score >= THRESHOLD else "sin_coincidencia"
        rows.append({
            "modelo": model_name,
            "consulta": query,
            "esperado": expected,
            "predicho": predicted,
            "producto_top": best["producto"],
            "score": round(best_score, 4),
            "acierto": predicted == expected,
        })
        if first_output_type is None:
            first_output_type = type(extractor(format_input(model_id, query, "query"))).__name__
    elapsed = time.perf_counter() - started
    return pd.DataFrame(rows), elapsed, first_output_type

local_results = []
local_timings = []
for model_name, model_id in MODEL_IDS.items():
    results, elapsed, output_type = evaluate_local(model_id, model_name)
    local_results.append(results)
    local_timings.append({"modelo": model_name, "segundos": round(elapsed, 2), "tipo_pipeline": output_type})

local_df = pd.concat(local_results, ignore_index=True)
display(pd.DataFrame(local_timings))
display(local_df)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 976.12it/s]


,modelo,segundos,tipo_pipeline
0,MiniLM-L12 multilingüe,0.90,list
1,E5 multilingüe small,0.46,list


,modelo,consulta,esperado,predicho,producto_top,score,acierto
0,MiniLM-L12 multilingüe,leche entera de un litro,leche,leche,Leche entera La Serenisima sachet 1 litro,0.6780,True
1,MiniLM-L12 multilingüe,leche para el desayuno,leche,sin_coincidencia,Leche entera La Serenisima sachet 1 litro,0.3744,False
2,MiniLM-L12 multilingüe,"yerba para mate, un kilo",yerba,yerba,Yerba Taragui con palo 1 kg,0.8511,True
3,MiniLM-L12 multilingüe,yerba mate suave,yerba,yerba,Yerba mate suave 500g,0.6969,True
4,MiniLM-L12 multilingüe,detergente para lavar los platos,detergente,sin_coincidencia,Leche entera La Serenisima sachet 1 litro,0.3292,False
5,MiniLM-L12 multilingüe,lavavajillas con limón,detergente,detergente,Lavavajillas Ala limon 750 ml,0.5754,True
6,MiniLM-L12 multilingüe,fideos tipo tirabuzón,fideos,sin_coincidencia,Yerba Taragui con palo 1 kg,0.4046,False
7,MiniLM-L12 multilingüe,arroz integral,sin_coincidencia,sin_coincidencia,Yerba mate suave 500g,0.4108,True
8,MiniLM-L12 multilingüe,jabón para manos,sin_coincidencia,sin_coincidencia,Detergente para vajilla 500ml,0.3272,True
9,MiniLM-L12 multilingüe,milk,leche,sin_coincidencia,Leche entera La Serenisima sachet 1 litro,0.3537,False


## Paso 5 — inferencia remota con `InferenceClient`

El cliente envía las mismas consultas, modelos y ofertas al proveedor `hf-inference`. Para E5, se aplican los prefijos `query:` y `passage:` recomendados por su model card; MiniLM recibe los textos sin prefijo. `pipeline` devuelve una lista de vectores y `InferenceClient.sentence_similarity` devuelve una lista de puntajes: revisamos `type()` y elegimos el puntaje del primer producto con `scores[best_index]`. Configurá un token con permisos de inferencia en una variable `HF_TOKEN` o en los secretos de Colab. **No imprimas el token ni lo guardes en una celda.** La celda conserva explícitamente errores de proveedor por consulta: si no hay acceso, anotá el error como tal y no inventes scores.

In [12]:
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except ImportError:
        pass
if not hf_token:
    raise RuntimeError("Definí HF_TOKEN como variable de entorno o secreto de Colab antes de llamar al proveedor.")

client = InferenceClient(provider="hf-inference", api_key=hf_token)
def evaluate_remote(model_id, model_name):
    rows = []
    started = time.perf_counter()
    first_output_type = None
    previous_error = None
    for query, expected in TEST_CASES:
        if previous_error:
            rows.append({
                "modelo": model_name,
                "consulta": query,
                "esperado": expected,
                "predicho": "no ejecutado",
                "producto_top": None,
                "score": np.nan,
                "acierto": False,
                "estado": "no ejecutado tras error previo",
                "error": previous_error,
            })
            continue

        sentence = query
        other_sentences = PRODUCTS
        if model_id == "intfloat/multilingual-e5-small":
            sentence = f"query: {query}"
            other_sentences = [f"passage: {product}" for product in PRODUCTS]

        try:
            scores = client.sentence_similarity(sentence=sentence, other_sentences=other_sentences, model=model_id)
        except Exception as exc:
            previous_error = f"{type(exc).__name__}: {exc}"
            rows.append({
                "modelo": model_name,
                "consulta": query,
                "esperado": expected,
                "predicho": "error de inferencia",
                "producto_top": None,
                "score": np.nan,
                "acierto": False,
                "estado": "error",
                "error": previous_error,
            })
            continue

        if first_output_type is None:
            first_output_type = type(scores).__name__
        best_index = int(np.argmax(scores))
        best = offers.iloc[best_index]
        best_score = float(scores[best_index])
        predicted = best["familia"] if best_score >= THRESHOLD else "sin_coincidencia"
        rows.append({
            "modelo": model_name,
            "consulta": query,
            "esperado": expected,
            "predicho": predicted,
            "producto_top": best["producto"],
            "score": round(best_score, 4),
            "acierto": predicted == expected,
            "estado": "ok",
            "error": None,
        })
    elapsed = time.perf_counter() - started
    return pd.DataFrame(rows), elapsed, first_output_type

remote_results = []
remote_timings = []
for model_name, model_id in MODEL_IDS.items():
    results, elapsed, output_type = evaluate_remote(model_id, model_name)
    remote_results.append(results)
    remote_timings.append({"modelo": model_name, "segundos": round(elapsed, 2), "tipo_inferenceclient": output_type})

remote_df = pd.concat(remote_results, ignore_index=True)
display(pd.DataFrame(remote_timings))
display(remote_df)

,modelo,segundos,tipo_inferenceclient
0,MiniLM-L12 multilingüe,1.12,None
1,E5 multilingüe small,0.68,None


,modelo,consulta,esperado,predicho,producto_top,score,acierto,estado,error
0,MiniLM-L12 multilingüe,leche entera de un litro,leche,error de inferencia,None,NaN,False,error,HfHubHTTPError: Client error '402 Payment Requ...
1,MiniLM-L12 multilingüe,leche para el desayuno,leche,no ejecutado,None,NaN,False,no ejecutado tras error previo,HfHubHTTPError: Client error '402 Payment Requ...
2,MiniLM-L12 multilingüe,"yerba para mate, un kilo",yerba,no ejecutado,None,NaN,False,no ejecutado tras error previo,HfHubHTTPError: Client error '402 Payment Requ...
3,MiniLM-L12 multilingüe,yerba mate suave,yerba,no ejecutado,None,NaN,False,no ejecutado tras error previo,HfHubHTTPError: Client error '402 Payment Requ...
4,MiniLM-L12 multilingüe,detergente para lavar los platos,detergente,no ejecutado,None,NaN,False,no ejecutado tras error previo,HfHubHTTPError: Client error '402 Payment Requ...
5,MiniLM-L12 multilingüe,lavavajillas con limón,detergente,no ejecutado,None,NaN,False,no ejecutado tras error previo,HfHubHTTPError: Client error '402 Payment Requ...
6,MiniLM-L12 multilingüe,fideos tipo tirabuzón,fideos,no ejecutado,None,NaN,False,no ejecutado tras error previo,HfHubHTTPError: Client error '402 Payment Requ...
7,MiniLM-L12 multilingüe,arroz integral,sin_coincidencia,no ejecutado,None,NaN,False,no ejecutado tras error previo,HfHubHTTPError: Client error '402 Payment Requ...
8,MiniLM-L12 multilingüe,jabón para manos,sin_coincidencia,no ejecutado,None,NaN,False,no ejecutado tras error previo,HfHubHTTPError: Client error '402 Payment Requ...
9,MiniLM-L12 multilingüe,milk,leche,no ejecutado,None,NaN,False,no ejecutado tras error previo,HfHubHTTPError: Client error '402 Payment Requ...


## Paso 6 — comparar y calibrar

Una diferencia entre `pipeline` e `InferenceClient` puede deberse al preprocesamiento, pooling, versiones o implementación del proveedor. E5 recibe los prefijos `query:` y `passage:` recomendados por su model card; MiniLM recibe los textos sin esos prefijos. Compará las predicciones consulta por consulta. El umbral `0.50` es solo un punto de partida: observá especialmente las consultas que deberían devolver **sin coincidencia**. Subí el umbral si acepta productos incorrectos; bajarlo puede recuperar más equivalencias, pero también aumentar falsos positivos.

In [13]:
all_results = pd.concat([
    local_df.assign(metodo="pipeline local"),
    remote_df.assign(metodo="InferenceClient remoto"),
], ignore_index=True)
comparison = all_results.pivot_table(
    index=["modelo", "consulta", "esperado"],
    columns="metodo",
    values=["predicho", "producto_top", "score", "acierto"],
    aggfunc="first",
)
comparison

acierto  \
metodo                                                                   InferenceClient remoto   
modelo                 consulta                         esperado                                  
E5 multilingüe small   arroz integral                   sin_coincidencia                  False   
                       detergente para lavar los platos detergente                        False   
                       fideos tipo tirabuzón            fideos                            False   
                       jabón para manos                 sin_coincidencia                  False   
                       lavavajillas con limón           detergente                        False   
                       leche entera de un litro         leche                             False   
                       leche para el desayuno           leche                             False   
                       milk                             leche                             False   
                       té                               sin_coincidencia                  False   
                       yerba mate suave                 yerba                             False   
                       yerba para mate, un kilo         yerba                             False   
MiniLM-L12 multilingüe arroz integral                   sin_coincidencia                  False   
                       detergente para lavar los platos detergente                        False   
                       fideos tipo tirabuzón            fideos                            False   
                       jabón para manos                 sin_coincidencia                  False   
                       lavavajillas con limón           detergente                        False   
                       leche entera de un litro         leche                             False   
                       leche para el desayuno           leche                             False   
                       milk                             leche                             False   
                       té                               sin_coincidencia                  False   
                       yerba mate suave                 yerba                             False   
                       yerba para mate, un kilo         yerba                             False   

                                                                                         \
metodo                                                                   pipeline local   
modelo                 consulta                         esperado                          
E5 multilingüe small   arroz integral                   sin_coincidencia          False   
                       detergente para lavar los platos detergente                 True   
                       fideos tipo tirabuzón            fideos                     True   
                       jabón para manos                 sin_coincidencia          False   
                       lavavajillas con limón           detergente                 True   
                       leche entera de un litro         leche                      True   
                       leche para el desayuno           leche                      True   
                       milk                             leche                      True   
                       té                               sin_coincidencia          False   
                       yerba mate suave                 yerba                      True   
                       yerba para mate, un kilo         yerba                      True   
MiniLM-L12 multilingüe arroz integral                   sin_coincidencia           True   
                       detergente para lavar los platos detergente                False   
                       fideos tipo tirabuzón            fideos                    False   
                       jabón para manos                 sin_coinc

In [15]:
negative_queries = {query for query, expected in TEST_CASES if expected == "sin_coincidencia"}
threshold_rows = []
threshold_sources = {
    "pipeline local": local_df.assign(estado="ok"),
    "InferenceClient remoto": remote_df,
}
for method, frame in threshold_sources.items():
    for model_name in MODEL_IDS:
        subset = frame[(frame["modelo"] == model_name) & (frame["consulta"].isin(negative_queries))]
        evaluated = subset[subset["estado"] == "ok"]
        for threshold in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
            false_matches = int((evaluated["score"] >= threshold).sum())
            threshold_rows.append({"metodo": method, "modelo": model_name, "umbral": threshold, "falsos_positivos_en_negativos": false_matches, "casos_negativos_evaluados": len(evaluated), "errores_o_no_ejecutados": len(subset) - len(evaluated)})
pd.DataFrame(threshold_rows)

,metodo,modelo,umbral,falsos_positivos_en_negativos,casos_negativos_evaluados,errores_o_no_ejecutados
0,pipeline local,MiniLM-L12 multilingüe,0.3,3,3,0
1,pipeline local,MiniLM-L12 multilingüe,0.4,1,3,0
2,pipeline local,MiniLM-L12 multilingüe,0.5,0,3,0
3,pipeline local,MiniLM-L12 multilingüe,0.6,0,3,0
4,pipeline local,MiniLM-L12 multilingüe,0.7,0,3,0
5,pipeline local,MiniLM-L12 multilingüe,0.8,0,3,0
6,pipeline local,E5 multilingüe small,0.3,3,3,0
7,pipeline local,E5 multilingüe small,0.4,3,3,0
8,pipeline local,E5 multilingüe small,0.5,3,3,0
9,pipeline local,E5 multilingüe small,0.6,3,3,0


## Conclusión experimental — ejecución local y remota

La prueba local se ejecutó en VS Code con las 11 consultas y el CSV de demostración (5 productos; precios inventados). Con umbral inicial 0,50, la línea base literal acertó 1/11, MiniLM 7/11 y E5 8/11. MiniLM acertó las tres consultas negativas; E5 confundió las tres negativas (`arroz integral`, `jabón para manos` y `té`) con productos existentes. Por eso no se declara un ganador: la muestra es pequeña y artificial, y el resultado de E5 muestra que el umbral 0,50 no evita falsos positivos en estos casos.

La llamada de prueba a `InferenceClient` se realizó para ambos modelos, pero el proveedor respondió **HTTP 402 Payment Required**: la cuenta no tenía créditos restantes. No se obtuvieron puntajes remotos; las tablas registran el error y las consultas no ejecutadas, no son una comparación válida de scores remotos. El token se leyó desde `.env` local, excluido de Git, y no se guardó en esta notebook.

La elección de E5 para la web no significa que haya sido el ganador demostrado por la pequeña prueba. Se lo eligió por su adecuación a recuperación semántica consulta-oferta, español y prefijos documentados; la evaluación muestra que hay falsos positivos y por eso el código exige también palabras clave. En la web se usan las ofertas manuales sincronizadas como fuente; la notebook solo usa el CSV artificial para una comparación reproducible. Los pesos se descargan del Hub, mientras que la inferencia de la web corre localmente en la API. **No se hizo entrenamiento ni ajuste del modelo.**

La web actual se probó en [http://127.0.0.1:8001/](http://127.0.0.1:8001/), que es una dirección local y no sirve desde otra computadora. No existe todavía un enlace público confirmado; la app integrada debe desplegarse y probarse para completar ese punto. La web usa FastAPI/HTML/JavaScript, no Gradio ni Streamlit; la demo Gradio separada usa datos ficticios y no sustituye la aplicación integrada. El chatbot Boti tampoco se da por terminado: sus opciones aún requieren corregirse y probarse.

Para completar la evaluación, repetí la prueba con ofertas reales observadas y suficientes productos por categoría. Si se habilitan créditos para el proveedor, volvé a ejecutar la celda de `InferenceClient` y compará sus resultados con el pipeline local. No publiques los precios del CSV como reales.

### Preguntas para discutir

- ¿Qué modelo acertó más en las consultas españolas? ¿Qué errores cometió cada uno?
- ¿Qué umbral elegiste y qué falsos positivos/negativos observaste?
- ¿Qué tiempos y tipos de salida obtuviste con `pipeline` y `InferenceClient`?
- ¿El proveedor respondió para ambos modelos? Anotá errores HTTP, permisos, cuotas o latencia si los hubo.
- ¿Qué limitaciones tiene evaluar con pocos productos y consultas escritas por una sola persona?

### Lista de evidencia para el informe

- Model cards consultadas y fecha de consulta.
- Resultado de las 11 consultas (incluidos los dos casos desafiantes `milk` y `té`), scores, etiquetas y tipos de retorno.
- Aclarar que `ofertas.csv` contiene datos de demostración; reemplazarlos por observaciones reales antes de presentar precios como reales.
- Tabla de resultados completa, incluidos los casos que fallaron.
- Umbral final y explicación de cómo lo calibraste.
- URL pública de la app y captura de una prueba, una vez desplegada.
- Decisiones de arquitectura: código en GitHub; pesos del modelo en Hugging Face; inferencia local en la API (y, una vez desplegada, en el servidor elegido). No afirmar que la inferencia de la web usa `InferenceClient`.
- Enlace público real de la app integrada y enlace a la carpeta GitHub del TP. El `127.0.0.1:8001` es solo local.
- Estado del requisito Gradio/Streamlit y del chatbot: la interfaz propia no es Gradio/Streamlit y las opciones del chat siguen pendientes de corrección.
- Informe `TP_Blanco_Griselda.pdf` y presentación `TP_Apellido_Nombre.pptx`, con decisiones, evidencia, limitaciones y pasos de reproducción.